# Day 25 — participant-level QC: 사람 단위로 세고 점검하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-4/day-25.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-4/day-25.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1: 가상 참가자 12명으로 QC 한 바퀴

실제 데이터 없이 전체 흐름을 연습합니다. 일부러 문제 두 개를 심어 두었습니다.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)

# 가상 참가자 12명: PD 7명, HC 5명 (실제 데이터가 아님!)
participants = pd.DataFrame({
    "participant_id": [f"S{i:02d}" for i in range(1, 13)],
    "group": ["PD"] * 7 + ["HC"] * 5,
    "age": rng.integers(55, 80, 12).astype(float),
})
participants.loc[3, "age"] = np.nan            # 나이 결측 1명

# 기록 표: 참가자 × 과제 3개 × 손목 2개 = 한 사람당 6줄이어야 정상
tasks = ["Relaxed", "StretchHold", "TouchNose"]
records = pd.DataFrame(
    [(p, t, w) for p in participants["participant_id"] for t in tasks for w in ["Left", "Right"]],
    columns=["participant_id", "task", "wrist"],
)
records["n_samples"] = 1024
# 일부러 문제를 심는다: S05는 TouchNose 기록이 없고, S09는 한 기록이 짧다
records = records[~((records["participant_id"] == "S05") & (records["task"] == "TouchNose"))]
records.loc[(records["participant_id"] == "S09") & (records["task"] == "Relaxed")
            & (records["wrist"] == "Left"), "n_samples"] = 600

# ↓↓↓ 실제 데이터로 바꿀 때는 위 두 표 대신 내 데이터의 참가자 표·기록 표를 읽어 온다 ↓↓↓

print("① 참가자 ID 중복:", participants["participant_id"].duplicated().sum())
print("② 군별 '사람' 수:\n", participants["group"].value_counts().to_string())
print("③ 군별 '기록' 수:\n", records.merge(participants)["group"].value_counts().to_string())
print("④ 참가자 표의 결측:\n", participants.isna().sum().to_string())

qc = (records.groupby("participant_id")
      .agg(n_records=("task", "size"), n_tasks=("task", "nunique"), min_samples=("n_samples", "min"))
      .reset_index()
      .merge(participants[["participant_id", "group"]]))
qc["complete"] = (qc["n_records"] == 6) & (qc["min_samples"] == 1024)
print("⑤ 문제가 있는 참가자:\n", qc[~qc["complete"]].to_string(index=False))
print("⑥ 완전한 참가자 수(군별):\n", qc[qc["complete"]]["group"].value_counts().to_string())

### 예시 3: 실제 PADS 참가자 표로 본 군 구성

PADS의 `preprocessed/file_list.csv`는 "한 사람 = 한 줄"인 참가자 표입니다. 아래 코드는 이 파일을 받아 둔 경우에 실행됩니다. 출력은 2026-11-12에 실제 파일로 실행해 얻은 값입니다. 여러분의 결과와 같은지 비교하세요.

In [ ]:
# 이 셀은 Day 24(PADS 데이터 내려받기)의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
import pandas as pd

path = "data/physionet_pd/pads/preprocessed/file_list.csv"   # 내 경로에 맞게 수정
df = pd.read_csv(path, dtype={"id": str})                    # ID는 문자열로!

print(df.shape)
print("ID 중복 없음:", df["id"].is_unique)
print(df["condition"].value_counts().to_string())
print(df["label"].value_counts().to_string())
print(pd.crosstab(df["label"], df["gender"]).to_string())

## 직접 해보기

1. `notebooks/04_pd_data_qc.ipynb`를 만들고 예시 1을 그대로 실행해 흐름을 익힙니다.
2. **참가자 표 읽기.** Day 24에서 고른 데이터의 참가자 표를 읽습니다.
   - PADS: 예시 3의 코드.
   - Gait: `demographics.txt`는 탭으로 구분되어 있지만 줄마다 끝에 붙은 빈 칸 수가 달라서 그냥 읽으면 `ParserError`가 납니다(실제로 확인함). 앞의 20개 열만 읽고, ID가 빈 줄을 지웁니다.

In [ ]:
df = pd.read_csv("<경로>/demographics.txt", sep="\t", usecols=range(20))
df = df.dropna(subset=["ID"])
print(df.shape)          # 실제 파일로 확인한 값: (166, 20)
print(df["Group"].value_counts())

Group·Gender의 숫자 코드가 무엇을 뜻하는지는 `demographics.html`(글자로 적혀 있음)과 대조해 데이터 사전에 적습니다.
3. **① ID 점검.** `df["<ID 열>"].is_unique`를 확인합니다. 신호 파일 이름에서 ID를 뽑아 참가자 표와 비교합니다.

In [ ]:
from pathlib import Path
files = sorted(Path("<신호 파일 폴더>").glob("*.txt"))        # 자리표시자
file_ids = {f.name.split("_")[0] for f in files}            # 파일 이름 규칙은 Day 24 데이터 사전 참고
table_ids = set(df["<ID 열>"])
print("표에만 있는 ID:", len(table_ids - file_ids), " 파일에만 있는 ID:", len(file_ids - table_ids))